In [0]:
%sql

create schema if not exists healthcare_endtoend_dev.bronze
Managed location 'abfss://unity-catalog-root@stazdatabricksendtoend.dfs.core.windows.net/healthcare-endtoend-dev/app-data/bronze'

In [0]:

source_path = "abfss://unity-catalog-root@stazdatabricksendtoend.dfs.core.windows.net/healthcare-endtoend-dev/stagingdata/diagnosis/"
checkpoint_path = "abfss://unity-catalog-root@stazdatabricksendtoend.dfs.core.windows.net/healthcare-endtoend-dev/bronze/diagnosis_raw/checkpoint/"
schema_location = "abfss://unity-catalog-root@stazdatabricksendtoend.dfs.core.windows.net/healthcare-endtoend-dev/bronze/diagnosis_raw/schema/"

# Autoloader read
df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("cloudFiles.maxFilesPerTrigger", 1) # READ ONE FILE AT A TIME
          .option("cloudFiles.schemaLocation", schema_location)  
          .load(source_path)
     )

# Write Bronze table (append)
(
    df.drop("_rescued_data")
      .writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .outputMode("append")
      .trigger(availableNow=True)   # AVAILABLE NOW → ingests all files & stops
      .toTable("healthcare_endtoend_dev.bronze.diagnosis_raw")
)

In [0]:
%sql

select * from healthcare_endtoend_dev.bronze.diagnosis_raw

In [0]:
import pandas as pd
import random
from datetime import datetime, timedelta

# Configuration
num_visits = 2500
patient_ids = [f"P{i:04d}" for i in range(1, 1001)]
hospital_ids = [f"H{i:03d}" for i in range(1, 21)]
diagnosis_codes = ["D001", "D002", "D003", "D004", "D005", "D006", "D007"]

def generate_dates():
    # Generate an admission date in 2025
    start_date = datetime(2025, 1, 1)
    admission = start_date + timedelta(days=random.randint(0, 360))
    # Discharge is 1 to 14 days later
    discharge = admission + timedelta(days=random.randint(1, 14))
    return admission.strftime('%Y-%m-%d'), discharge.strftime('%Y-%m-%d')

# Generate Data
visits_data = []
for i in range(1, num_visits + 1):
    adm, dis = generate_dates()
    visits_data.append({
        "visit_id": f"V{i:04d}",
        "patient_id": random.choice(patient_ids),
        "hospital_id": random.choice(hospital_ids),
        "admission_date": adm,
        "discharge_date": dis,
        "diagnosis_code": random.choice(diagnosis_codes),
        "cost": round(random.uniform(5000, 50000), 2)
    })

# Create DataFrame
df_visits = pd.DataFrame(visits_data)

# Show first few rows to verify format
display(df_visits.head(1000))
